first: get access to the parquet
second: put all genres and themes into a set, creating a vocabulary for matrix later
third: sort set, and map out all tags to an index/column number
fourth: make a matrix, 4812 x 73 size, which is # of anime x # of tags. all values are 0
fifth: iterate through every anime, and for their corresponding genres/themes, make those values 1 in the matrix

In [9]:
import pandas as pd
import numpy as np

df = pd.read_parquet(path="../data/anime.parquet")

vocabulary = set()

# for every anime, add the genres and themes into a set, to create the vocabulary needed for the vector matrix
for a in df["genres"]:
    for genre in a:
        vocabulary.add(genre)
for a in df["themes"]:
    for theme in a:
        vocabulary.add(theme)


sorted_vocab = sorted(vocabulary)

# create dict mapping to all different genres/themes
tag_map = {}
for n, t in enumerate(sorted_vocab):
    tag_map[t] = n

# make matrix
matrix = np.zeros((len(df),len(sorted_vocab)), dtype=np.float32)

# every anime ordered from 0-4800 approx
for i in range(len(df["title"])):
    genres = df["genres"][i] # get the genres, by accessing the genres at that specific index, returns a list
    # iterate the list of genres for said anime, then in the matrix, map to the specific anime and then tag_map to find the column of that tag
    for g in genres:
        matrix[i][tag_map[g]] = 1
    themes = df["themes"][i]
    for t in themes:
        matrix[i][tag_map[t]] = 1

def find_title(name: str):
    return df[df["title"].str.contains(name, case=False, na=False)]


title_to_row = {}
for i, id in enumerate(df["title"]):
    title_to_row[id] = i

# prefs example (3 inputs)
rows = []
for t in ["Mob Psycho 100", "Attack on Titan Season 3 Part 2", "Blue Lock"]:
    rows.append(matrix[title_to_row[t]])

taste_vector = np.mean(rows, axis=0) # just adding all separate vectors together with their corresponding section, then averaging by # of inputs


# cosine similarity formula

# does dot product between matrix and taste_vector, as well as the sum, so every value is now similarity btw prefs and said anime
numerators = matrix @ taste_vector

# denominators np func basically just finds vector length using sqrt(sum of squares), we find for both and we multiply em 
taste_norm = np.linalg.norm(taste_vector)
anime_norms = np.linalg.norm(matrix, axis=1) # matrix is a list of the anime vector embeddings, so each one has a different vector length

# division
scores = numerators / (taste_norm * anime_norms)

print(scores.max())
print(scores.min())



0.79259396
0.0
